# Build stock and news dataframes

Run after `collect_data.ipynb` and `clean_data.ipynb`. Following the EDA, use AAPL and JNJ. Keep the daily stock fields and add close lags for the previous one, two, and three trading days per ticker. Each lag is missing where the ticker has insufficient history.

Match articles to the ticker and publication date in New York time. Average each FinBERT probability across articles on the same ticker-day. Articles on dates without a matching trading bar are omitted. The publication date includes after-close articles, so these scores represent the full calendar day.

In [5]:
import os
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

os.environ["USE_TF"] = "0"
from transformers import pipeline

data_dir = Path("../data") if Path.cwd().name == "src" else Path("data")
prices = pd.read_parquet(data_dir / "raw/prices.parquet")
articles = pd.read_parquet(data_dir / "processed/news_clean.parquet")

df_original = prices.loc[prices["symbol"].isin(["AAPL", "JNJ"])].copy()
df_original["timestamp"] = pd.to_datetime(df_original["timestamp"], utc=True)
df_original = df_original.sort_values(["symbol", "timestamp"]).reset_index(drop=True)
close_by_symbol = df_original.groupby("symbol")["close"]
for lag in range(1, 4):
    df_original[f"close_lag_{lag}"] = close_by_symbol.shift(lag)
df_original.head()

,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,close_lag_1,close_lag_2,close_lag_3
0,AAPL,2020-01-02 05:00:00+00:00,71.28,72.33,71.03,72.27,146217352.0,283489.0,71.84,NaN,NaN,NaN
1,AAPL,2020-01-03 05:00:00+00:00,71.50,72.33,71.35,71.57,154711256.0,280441.0,71.91,72.27,NaN,NaN
2,AAPL,2020-01-06 05:00:00+00:00,70.69,72.18,70.44,72.14,125599968.0,242868.0,71.66,71.57,72.27,NaN
3,AAPL,2020-01-07 05:00:00+00:00,72.15,72.40,71.58,71.80,118227136.0,227871.0,71.96,72.14,71.57,72.27
4,AAPL,2020-01-08 05:00:00+00:00,71.50,73.26,71.50,72.95,142736804.0,261773.0,72.50,71.80,72.14,71.57


In [ ]:
import warnings
from bs4 import XMLParsedAsHTMLWarning
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

labels = ["negative", "neutral", "positive"]
keys = ["symbol", "timestamp"]

articles = articles.rename(columns={"ticker": "symbol"})
articles["timestamp"] = (
    pd.to_datetime(articles["created_at"], utc=True)
    .dt.tz_convert("America/New_York")
    .dt.normalize()
    .dt.tz_convert("UTC")
    .astype(df_original["timestamp"].dtype)
)
articles = articles.merge(df_original[keys], on=keys, validate="many_to_one")
articles["text"] = articles["text_for_sentiment"].fillna("").map(
    lambda text: BeautifulSoup(text, "html.parser").get_text(" ", strip=True)
)
articles = articles.loc[articles["text"].ne("")].reset_index(drop=True)

scores_path = data_dir / "processed/article_sentiment.parquet"

if scores_path.exists():
    article_scores = pd.read_parquet(scores_path)
else:
    sentiment = pipeline(
        "text-classification", model="ProsusAI/finbert", framework="pt", device="cpu"
    )
    results = sentiment(
        articles["text"].tolist(), truncation=True, top_k=None, batch_size=8
    )
    scores = pd.DataFrame(
        [{item["label"]: item["score"] for item in result} for result in results]
    )[labels]
    article_scores = articles[keys].join(scores)
    article_scores.to_parquet(scores_path)

daily_sentiment = article_scores.groupby(keys)[labels].mean()
df_news = df_original.join(daily_sentiment, on=keys, validate="one_to_one")
df_news[labels] = df_news[labels].fillna(0.0)

pd.testing.assert_frame_equal(df_news[df_original.columns], df_original)
has_news = pd.MultiIndex.from_frame(df_original[keys]).isin(daily_sentiment.index)
assert df_news.loc[~has_news, labels].eq(0.0).all().all()
assert df_news.loc[has_news, labels].sum(axis=1).sub(1.0).abs().lt(1e-5).all()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,close_lag_1,close_lag_2,close_lag_3,negative,neutral,positive
0,AAPL,2020-01-02 05:00:00+00:00,71.28,72.33,71.03,72.27,146217352.0,283489.0,71.84,NaN,NaN,NaN,0.336943,0.459312,0.203745
1,AAPL,2020-01-03 05:00:00+00:00,71.50,72.33,71.35,71.57,154711256.0,280441.0,71.91,72.27,NaN,NaN,0.202425,0.180871,0.616704
2,AAPL,2020-01-06 05:00:00+00:00,70.69,72.18,70.44,72.14,125599968.0,242868.0,71.66,71.57,72.27,NaN,0.360663,0.376143,0.263194
3,AAPL,2020-01-07 05:00:00+00:00,72.15,72.40,71.58,71.80,118227136.0,227871.0,71.96,72.14,71.57,72.27,0.017109,0.675395,0.307496
4,AAPL,2020-01-08 05:00:00+00:00,71.50,73.26,71.50,72.95,142736804.0,261773.0,72.50,71.80,72.14,71.57,0.017891,0.294529,0.687579


## Verify columns and inspect sample values

In [8]:
df_news.head()

,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,close_lag_1,close_lag_2,close_lag_3,negative,neutral,positive
0,AAPL,2020-01-02 05:00:00+00:00,71.28,72.33,71.03,72.27,146217352.0,283489.0,71.84,NaN,NaN,NaN,0.336943,0.459312,0.203745
1,AAPL,2020-01-03 05:00:00+00:00,71.50,72.33,71.35,71.57,154711256.0,280441.0,71.91,72.27,NaN,NaN,0.202425,0.180871,0.616704
2,AAPL,2020-01-06 05:00:00+00:00,70.69,72.18,70.44,72.14,125599968.0,242868.0,71.66,71.57,72.27,NaN,0.360663,0.376143,0.263194
3,AAPL,2020-01-07 05:00:00+00:00,72.15,72.40,71.58,71.80,118227136.0,227871.0,71.96,72.14,71.57,72.27,0.017109,0.675395,0.307496
4,AAPL,2020-01-08 05:00:00+00:00,71.50,73.26,71.50,72.95,142736804.0,261773.0,72.50,71.80,72.14,71.57,0.017891,0.294529,0.687579


In [7]:
expected_stock_columns = [
    "symbol", "timestamp", "open", "high", "low", "close", "volume",
    "trade_count", "vwap", "close_lag_1", "close_lag_2", "close_lag_3",
]
assert df_original.columns.tolist() == expected_stock_columns
assert df_news.columns.tolist() == expected_stock_columns + ["negative", "neutral", "positive"]

for lag in (1, 2, 3):
    expected_lag = df_original.groupby("symbol")["close"].shift(lag)
    pd.testing.assert_series_equal(df_original[f"close_lag_{lag}"], expected_lag, check_names=False)
assert df_news[labels].ge(0).all().all() and df_news[labels].le(1).all().all()
news_rows = df_news.set_index(keys).loc[daily_sentiment.index, labels]
pd.testing.assert_frame_equal(news_rows, daily_sentiment)

for name, frame in [("df_original", df_original), ("df_news", df_news)]:
    print(f"{name} ({len(frame):,} rows): {frame.columns.tolist()}")

sample_columns = keys + ["close", "close_lag_1", "close_lag_2", "close_lag_3"] + labels
print(f"With scored news ({has_news.sum():,} rows)")
display(df_news.loc[has_news, sample_columns].head(3))
print(f"Without scored news ({(~has_news).sum():,} rows)")
display(df_news.loc[~has_news, sample_columns].head(3))
print("All dataframe checks passed.")

df_original (3,412 rows): ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap', 'close_lag_1', 'close_lag_2', 'close_lag_3']
df_news (3,412 rows): ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap', 'close_lag_1', 'close_lag_2', 'close_lag_3', 'negative', 'neutral', 'positive']
With scored news (3,066 rows)


,symbol,timestamp,close,close_lag_1,close_lag_2,close_lag_3,negative,neutral,positive
0,AAPL,2020-01-02 05:00:00+00:00,72.27,NaN,NaN,NaN,0.336943,0.459312,0.203745
1,AAPL,2020-01-03 05:00:00+00:00,71.57,72.27,NaN,NaN,0.202425,0.180871,0.616704
2,AAPL,2020-01-06 05:00:00+00:00,72.14,71.57,72.27,NaN,0.360663,0.376143,0.263194


Without scored news (346 rows)


,symbol,timestamp,close,close_lag_1,close_lag_2,close_lag_3,negative,neutral,positive
80,AAPL,2020-04-28 04:00:00+00:00,67.19,68.30,68.25,66.34,0.0,0.0,0.0
87,AAPL,2020-05-07 04:00:00+00:00,73.26,72.51,71.77,70.71,0.0,0.0,0.0
102,AAPL,2020-05-29 04:00:00+00:00,76.89,76.97,76.93,76.60,0.0,0.0,0.0


All dataframe checks passed.


In [9]:
print(df_original.columns)
print(df_news.columns)

Index(['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume',
       'trade_count', 'vwap', 'close_lag_1', 'close_lag_2', 'close_lag_3'],
      dtype='str')
Index(['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume',
       'trade_count', 'vwap', 'close_lag_1', 'close_lag_2', 'close_lag_3',
       'negative', 'neutral', 'positive'],
      dtype='str')
